# Enhanced Symbolic Discovery Engine (ESDE)

A hybrid neural-symbolic system for solving mathematical, logical, and geometric problems, targeting AIME, SAT, and GSM8K benchmarks. Integrates neural fields for spatial reasoning and real-world 3D scenarios.

## Features
- Parses natural language into structured sketches.
- Generates and verifies algebraic and geometric rules.
- Evolves rules using contrastive learning.
- Embeds expressions for semantic retrieval and pattern discovery.
- Models 3D geometries with neural fields.
- Stores knowledge for meta-learning.
- Refines outputs for clarity.

## Dependencies
- Python 3.8+
- `pydantic`, `sympy`, `z3-solver`, `transformers`, `cma`, `scikit-learn`, `mlxtend`, `torch`, `tinycudann`, `numpy`, `networkx`, `faiss-cpu`, `pytest`, `json_repair`, `redis`

## Setup
Run in a Jupyter environment with dependencies installed:
```bash
pip install pydantic sympy z3-solver transformers cma scikit-learn mlxtend torch tinycudann numpy networkx faiss-cpu pytest json_repair redis

In [47]:
### **Imports and Setup (Code)**

# Import dependencies for ESDE functionality
import json
import re
import traceback
import requests
import numpy as np
from typing import List, Dict, Union
from pydantic import BaseModel, ValidationError
from sympy import sympify, Equality, Add, Mul, Sum, symbols, simplify, Expr
from sympy.abc import k, n, x, i
from sympy.core.relational import Relational
from sympy.parsing.sympy_parser import parse_expr
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
import networkx as nx
import pandas as pd
from mlxtend.frequent_patterns import fpgrowth
import torch
import faiss
import json_repair
import redis
from concurrent.futures import ThreadPoolExecutor
import asyncio
from z3 import Solver, Int, sat

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings("ignore")

try:
    import tinycudann as tcnn
    HAS_TCNN = True
except ModuleNotFoundError:
    HAS_TCNN = False


# Set random seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Ollama wrapper for local Mistral model
def call_ollama(prompt: str, model: str = "mistral", temperature: float = 0.2) -> str:
    """Call the Ollama API to generate text using a local model."""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False, "temperature": temperature}
    )
    return response.json()["response"].strip()

# Redis cache for LLM outputs
# ---- Optional Redis cache -------------------------------------------------
try:
    import redis
    _tmp_client = redis.Redis(host="localhost", port=6379, db=0, socket_connect_timeout=0.5)
    _tmp_client.ping()
    cache: "RedisLike" = _tmp_client          # type: ignore
except Exception:
    class _NullCache(dict):                   # fake drop-in replacement
        def get(self, k):       return None
        def set(self, k, v):    pass
    cache = _NullCache()
# ---------------------------------------------------------------------------

## Multi-Agent Sketch Negotiator

**Purpose**: Parses natural language problem statements into structured JSON sketches, detecting spatial tasks for routing to the geometry module.

**Technical Details**:
- Uses Phi-3 (via Ollama) to propose sketches, SymPy/Z3 to validate, and rule-based refinement.
- Employs CMA-ES for optimization (simplified here as retries).
- Pydantic ensures robust JSON validation.

In [27]:
class Sketch(BaseModel):
    """Pydantic model for structured problem sketches."""
    task_type: str
    domain: str
    goal: str
    axioms: List[str]
    method: str

class SketchNegotiator:
    """Negotiates a structured sketch from a natural language problem."""
    def __init__(self, model_name: str = "mistral", max_retries: int = 2):
        self.model_name = model_name
        self.max_retries = max_retries

    def propose_sketch(self, problem: str) -> Sketch:
        """Propose a structured sketch from a problem statement."""
        # Check cache first
        cache_key = f"sketch:{hash(problem)}"
        cached = cache.get(cache_key)
        if cached:
            print("Using cached sketch")
            return Sketch.model_validate_json(cached)

        prompt = f"""You are a proof assistant. Convert the math problem into a structured sketch.
        Return a JSON object with:
        - task_type
        - domain
        - goal
        - axioms (list)
        - method (string or list)
        Use only arithmetic/algebraic language. No comments.
        Problem: {problem}"""

        for attempt in range(1, self.max_retries + 1):
            raw = call_ollama(prompt, model=self.model_name)
            print(f"[Attempt {attempt}] LLM raw output:\n{raw}\n")
            try:
                repaired = json_repair.repair_json(raw)
                sketch = self._parse_json(repaired)
                if self._validate_sketch(sketch) and self._check_logical_consistency(sketch):
                    cache.set(cache_key, sketch.model_dump_json())
                    return sketch
            except Exception as e:
                print(f"Attempt {attempt} failed: {e}")
        return self._fallback_sketch()

    def _parse_json(self, text: str) -> Sketch:
        """Parse and validate JSON from LLM output."""
        match = re.search(r"\{.*?\}", text, re.DOTALL)
        if not match:
            raise ValueError("No JSON block found")
        block = match.group(0)
        block = re.sub(r',\s*(?=[}\]])', '', block)
        obj = json.loads(block)
        if isinstance(obj.get("method"), list):
            obj["method"] = " \n".join(obj["method"])
        return Sketch.model_validate(obj)

    def _validate_sketch(self, sketch: Sketch) -> bool:
        """Validate sketch for minimum requirements."""
        return bool(sketch.axioms and len(sketch.axioms) >= 2 and sketch.method and len(sketch.method) >= 10)

    def _check_logical_consistency(self, sketch: Sketch) -> bool:
        """Check logical consistency of axioms using Z3."""
        solver = Solver()
        for axiom in sketch.axioms:
            try:
                solver.add(sympify(axiom))
            except:
                return False
        return solver.check() == sat

    def _fallback_sketch(self) -> Sketch:
        """Provide a fallback sketch if negotiation fails."""
        return Sketch(
            task_type="proof",
            domain="algebra",
            goal="Prove that the sum of the first n odd numbers is n^2",
            axioms=["An odd number is of the form 2*k - 1", "Sum of first n odd numbers = n^2"],
            method="induction"
        )

In [28]:
# Sanity Check for SketchNegotiator
negotiator = SketchNegotiator()
sketch = negotiator.propose_sketch("Prove that the sum of the first n odd numbers is n squared")
print("Sketch Output:", sketch.model_dump_json(indent=2))

ConnectionError: Error 111 connecting to localhost:6379. Connection refused.

## Neural-Symbolic Rule Engine

**Purpose**: Generates and executes rules for algebraic and geometric problems, blending neural creativity with symbolic precision.

**Technical Details**:
- Uses Phi-3 to propose rules in a DSL, executed by SymPy.
- Supports parallel rule generation for efficiency.

In [29]:
class RuleEngine:
    """Generates and executes symbolic rules for problem-solving."""
    def __init__(self, model_name: str = "mistral", max_candidates: int = 10, max_retries: int = 2):
        self.model_name = model_name
        self.max_candidates = max_candidates
        self.max_retries = max_retries
        self.templates = {
            "induction": "Eq(Sum(2*i - 1, (i, 1, n)), n**2)",
            "algebra": "Eq(a + b, c)"
        }

    def propose_rules(self, sketch: Sketch) -> List[str]:
        """Propose candidate rules based on the sketch."""
        prompt = f"""Generate {self.max_candidates} rules using templates: {self.templates}.
        Task: {sketch.task_type}, Goal: {sketch.goal}"""
        
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(lambda: call_ollama(prompt, self.model_name)) 
                       for _ in range(self.max_retries)]
            responses = [f.result() for f in futures]
        
        candidates = [self._extract_candidates(resp) for resp in responses]
        return self._clean_and_flatten([c for sublist in candidates for c in sublist])

    def _extract_candidates(self, text: str) -> List[str]:
        """Extract rule expressions from LLM output."""
        return [m.group(0) for ln in text.splitlines() if (m := re.search(r"Eq\(.*\)", ln))]

    def _clean_and_flatten(self, rules: List[str]) -> List[str]:
        """Clean and filter valid rule expressions."""
        out = []
        nested = re.compile(r"Eq\(\s*(.+?)\s*,\s*Eq\(\s*(.+?)\s*,\s*(.+?)\s*\)\s*\)")
        for r in rules:
            r0 = re.sub(r"\s+", " ", r)
            if "[" in r0 or "for " in r0 or "->" in r0: continue
            m = nested.match(r0)
            if m:
                r0 = f"Eq({m.group(1)}, {m.group(3)})"
            if re.match(r"^Eq\([^,]+,\s*[^,]+\)$", r0):
                out.append(r0)
        return out

    def execute_rules(self, rules: List[str], sketch: Sketch) -> List[Dict]:
        """Execute rules and validate their results."""
        steps = []
        for rule in rules:
            try:
                expr = sympify(rule)
                steps.append({"rule": rule, "result": str(expr), "valid": expr.is_Equality})
            except Exception as e:
                steps.append({"rule": rule, "result": str(e), "valid": False})
        return steps

In [30]:
# Sanity Check for RuleEngine
rule_engine = RuleEngine()
sketch = Sketch(task_type="proof", domain="algebra", goal="sum(2*k - 1, (k, 1, n)) = n**2", axioms=[], method="induction")
rules = rule_engine.propose_rules(sketch)
print("Generated Rules:", rules)

Generated Rules: ['Eq(3, 4)']


## Verifier-as-Loss Oracle

**Purpose**: Computes loss for rules to guide optimization.

**Technical Details**:
- Uses SymPy for simplification and Z3 for constraint validation.
- Calculates loss as squared differences or violation counts.

In [31]:
class Verifier:
    """Verifies rules and computes loss for optimization."""
    def score_rules(self, rules: List[str]) -> List[Dict]:
        """Score rules based on validity and loss."""
        results = []
        for rule in rules:
            rec = {"rule": rule, "valid": False, "loss": float('inf')}
            try:
                expr = sympify(self._repair_rule(rule))
                if isinstance(expr, Equality):
                    solver = Solver()
                    n = Int('n')
                    solver.add(expr.lhs == expr.rhs)
                    if solver.check() == sat:
                        rec.update(valid=True, loss=0.0)
                    else:
                        diff = simplify(expr.lhs - expr.rhs)
                        rec.update(loss=float(diff.evalf()**2) if diff.is_number else 1.0)
            except Exception as e:
                rec["error"] = str(e)
            results.append(rec)
        return results

    def _repair_rule(self, rule: str) -> str:
        """Repair malformed rule strings."""
        r = rule.replace("==", "=")
        r = re.sub(r"Eq\(\s*(.+?)\s*,\s*Eq\(\s*(.+?)\s*,\s*(.+?)\s*\)\s*\)", r"Eq(\1,\3)", r)
        r = re.sub(r"//.*", "", r)
        return r.strip()

In [32]:
# Sanity Check for Verifier
verifier = Verifier()
test_rules = ["Eq(Sum(2*i - 1, (i, 1, n)), n**2)", "Eq(Sum(2*i - 1, (i, 1, n)), n**2 + 1)"]
scores = verifier.score_rules(test_rules)
print("Rule Scores:", scores)

Rule Scores: [{'rule': 'Eq(Sum(2*i - 1, (i, 1, n)), n**2)', 'valid': False, 'loss': 0.0}, {'rule': 'Eq(Sum(2*i - 1, (i, 1, n)), n**2 + 1)', 'valid': False, 'loss': 1.0}]


## Rulial Contrastive Evolver

**Purpose**: Evolves rules by contrasting successful and unsuccessful applications.

**Technical Details**:
- Uses verifier scores to guide evolution.
- Applies crossover and mutation on symbolic expressions.

In [33]:
import random

class Evolver:
    """Evolves rules using contrastive learning."""
    def __init__(self, verifier):
        self.verifier = verifier

    def evolve(self, candidates: List[Dict], domain: str) -> List[Dict]:
        """Evolve rules over generations."""
        population = candidates[:]
        for _ in range(3):  # Generations
            new_pop = []
            for _ in range(len(population)):
                p1, p2 = random.sample(population, 2)
                child = self._crossover(p1, p2)
                child = self._mutate(child)
                new_pop.append(child)
            population = new_pop
        return [min(self.verifier.score_rules([c["rule"] for c in population]), 
                   key=lambda x: x["loss"])]

    def _crossover(self, p1: Dict, p2: Dict) -> Dict:
        """Combine parts of two rules."""
        lhs = sympify(p1["rule"].split(',')[0][3:])
        rhs = sympify(p2["rule"].split(',')[1][:-1])
        return {"rule": f"Eq({lhs}, {rhs})"}

    def _mutate(self, cand: Dict) -> Dict:
        """Mutate a rule slightly."""
        expr = sympify(cand["rule"])
        rhs = expr.rhs + random.randint(-2, 2)
        return {"rule": f"Eq({expr.lhs}, {rhs})"}

In [34]:
# Sanity Check for Evolver
evolver = Evolver(verifier)
near_misses = [{"rule": "Eq(Sum(2*i - 1, (i, 1, n)), n**2 + 1)", "valid": False, "loss": 1.0}]
evolved = evolver.evolve(near_misses, domain="algebra")
print("Evolved Rules:", evolved)

ValueError: Sample larger than population or is negative

## Symbolic Embedding and Clustering

**Purpose**: Embeds expressions for semantic retrieval and clustering.

**Technical Details**:
- Uses CodeBERT for semantic embeddings and K-means for clustering.
- Combines structural (tree depth) and semantic features.

In [35]:
from transformers import AutoTokenizer, AutoModel

class SymbolicEmbedding:
    """Embeds and clusters symbolic expressions."""
    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
        self.model = AutoModel.from_pretrained("microsoft/codebert-base")
        self.kmeans = KMeans(n_clusters=5)

    def tree_to_tokens(self, expr: Expr) -> List[str]:
        """Convert expression tree to token list."""
        tokens = []
        if hasattr(expr, "args"):
            for arg in expr.args:
                tokens.extend(self.tree_to_tokens(arg))
            tokens.append(str(expr.func))
        else:
            tokens.append(str(expr))
        return tokens

    def embed_expression(self, expr: str) -> tuple[float, np.ndarray]:
        """Embed an expression using CodeBERT."""
        inputs = self.tokenizer(expr, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = self.model(**inputs)
        embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
        return len(self.tree_to_tokens(sympify(expr))), embedding

    def cluster_expressions(self, exprs: List[str]) -> List[int]:
        """Cluster expressions based on embeddings."""
        embeddings = [self.embed_expression(e)[1] for e in exprs]
        return self.kmeans.fit_predict(embeddings).tolist()

In [36]:
# Sanity Check for Symbolic Embedding
embedder = SymbolicEmbedding()
test_exprs = ["sum(2*k - 1, (k, 1, n)) = n**2", "sum(2*k, (k, 1, n)) = n*(n+1)"]
depths, embeddings = zip(*[embedder.embed_expression(e) for e in test_exprs])
cluster_ids = embedder.cluster_expressions(test_exprs)
print("Cluster Assignments:", cluster_ids)

SympifyError: Sympify of expression 'could not parse 'sum(2*k - 1, (k, 1, n)) = n**2'' failed, because of exception being raised:
SyntaxError: invalid syntax (<string>, line 1)

## Memory Graph

**Purpose**: Stores and retrieves knowledge for meta-learning.

**Technical Details**:
- Uses FAISS for fast embedding search and NetworkX for relational storage.
- Mines patterns with FP-growth.

In [37]:
class MemoryGraph:
    """Stores and retrieves symbolic knowledge."""
    def __init__(self, embedder: SymbolicEmbedding, dim: int = 768):
        self.graph = nx.Graph()
        self.embedder = embedder
        self.index = faiss.IndexHNSWFlat(dim, 32)  # HNSW with 32 neighbors
        self.exprs = []
        self.metadata = []

    def store(self, expr: str, embedding: np.ndarray, metadata: Dict):
        """Store an expression and its metadata."""
        node_id = len(self.graph.nodes)
        self.graph.add_node(node_id, expr=expr, metadata=metadata)
        self.index.add(embedding.reshape(1, -1))
        self.exprs.append(expr)
        self.metadata.append(metadata)

    def retrieve(self, query_expr: str, k: int = 3) -> List[Dict]:
        """Retrieve similar expressions."""
        _, query_vec = self.embedder.embed_expression(query_expr)
        distances, indices = self.index.search(query_vec.reshape(1, -1), k)
        return [{"expr": self.exprs[i], "distance": distances[0][j], "metadata": self.metadata[i]} 
                for j, i in enumerate(indices[0])]

    def mine_patterns(self, exprs: List[str]) -> List[str]:
        """Mine frequent patterns from expressions."""
        token_lists = [self.embedder.tree_to_tokens(sympify(e)) for e in exprs]
        df = pd.DataFrame(token_lists).fillna(0)
        frequent_patterns = fpgrowth(df, min_support=0.5, use_colnames=True)
        return [str(p) for p in frequent_patterns["itemsets"]]

In [38]:
# Sanity Check for MemoryGraph
memory = MemoryGraph(embedder)
for expr in test_exprs:
    _, emb = embedder.embed_expression(expr)
    memory.store(expr, emb, {"sketch": sketch, "tag": "test_expr"})
retrieved = memory.retrieve(test_exprs[0], k=2)
print("Retrieved Neighbors:", retrieved)

SympifyError: Sympify of expression 'could not parse 'sum(2*k - 1, (k, 1, n)) = n**2'' failed, because of exception being raised:
SyntaxError: invalid syntax (<string>, line 1)

## Neural Field Geometry Module

**Purpose**: Models 3D geometries and extracts symbolic constraints.

**Technical Details**:
- Placeholder for Instant-NGP integration.
- Extracts symbolic constraints from problem statements.

## Feedback Loop

**Purpose**: Refines outputs for clarity and triggers improvements.

**Technical Details**:
- Evaluates output clarity and suggests enhancements.
- Triggers re-processing if quality is low.

In [41]:
class FeedbackLoop:
    """Refines outputs and provides feedback."""
    def evaluate_output(self, output: Dict) -> tuple[float, List[str]]:
        """Evaluate output clarity."""
        suggestions = []
        if not output.get("steps"): suggestions.append("1. Add detailed steps.")
        if len(output["proof"]) < 10: suggestions.append("2. Expand proof explanation.")
        score = 1.0 - (len(suggestions) / 5.0)
        return score, suggestions

    def trigger_improvements(self, output: Dict, loss: float) -> Dict:
        """Trigger refinements based on score and loss."""
        score, suggestions = self.evaluate_output(output)
        if score < 0.9 or loss > 0.0:
            output["annotations"] = suggestions
        return output

In [42]:
# Sanity Check for FeedbackLoop
feedback = FeedbackLoop()
test_output = {
    "proof": "sum(2*k - 1, (k, 1, n)) = n**2",
    "method": "induction",
    "steps": [],
    "status": "partial",
    "annotations": [],
    "cluster_id": 0
}
refined_output = feedback.trigger_improvements(test_output, 0.5)
print("Refined Output:", refined_output)

Refined Output: {'proof': 'sum(2*k - 1, (k, 1, n)) = n**2', 'method': 'induction', 'steps': [], 'status': 'partial', 'annotations': ['1. Add detailed steps.'], 'cluster_id': 0}


## Main ESDE Pipeline

**Purpose**: Orchestrates all ESDE modules to solve problems.

**Technical Details**:
- Integrates negotiation, rule generation, verification, evolution, embedding, memory, and feedback.
- Uses async for efficiency.

In [43]:
class ESDE:
    """Main pipeline for the Enhanced Symbolic Discovery Engine."""
    def __init__(self, model_name: str = "mistral"):
        self.negotiator = SketchNegotiator(model_name)
        self.rule_engine = RuleEngine(model_name)
        self.verifier = Verifier()
        self.evolver = Evolver(self.verifier)
        self.embedder = SymbolicEmbedding()
        self.memory = MemoryGraph(self.embedder)
        self.geometry = NeuralFieldGeometry()
        self.feedback = FeedbackLoop()

    async def solve(self, problem: str) -> Dict:
        """Solve a problem using the ESDE pipeline."""
        sketch = await asyncio.to_thread(self.negotiator.propose_sketch, problem)
        rules = await asyncio.to_thread(self.rule_engine.propose_rules, sketch)
        steps = self.rule_engine.execute_rules(rules, sketch)
        scores = self.verifier.score_rules(rules)
        loss_scores = [s["loss"] for s in scores]
        if max(loss_scores) >= 0.9:
            steps = await asyncio.to_thread(self.evolver.evolve, steps, sketch.domain)
        valid_exprs = [s["rule"] for s in steps if s["valid"]]
        for expr in valid_exprs:
            _, emb = self.embedder.embed_expression(expr)
            self.memory.store(expr, emb, {"sketch": sketch, "status": "valid"})
        output = {
            "proof": sketch.goal,
            "method": sketch.method,
            "steps": [s["result"] for s in steps if s["valid"]],
            "status": "verified" if any(s["valid"] for s in steps) else "partial",
            "annotations": [],
            "cluster_id": 0
        }
        loss = min(loss_scores) if loss_scores else 1.0
        return self.feedback.trigger_improvements(output, loss)

In [44]:
# Sanity Check for ESDE Pipeline
esde = ESDE()
algebraic_result = await esde.solve("Prove that the sum of the first n odd numbers is n squared")
print("Algebraic Result:", algebraic_result)

ConnectionError: Error 111 connecting to localhost:6379. Connection refused.

## Demonstrations

**Purpose**: Showcases ESDE on sample tasks.

**Tasks**:
1. Algebraic: "Prove that the sum of the first n odd numbers is n²"
2. Geometric: "Find the area of a triangle with vertices (0,0,0), (1,0,0), (0,1,0)"

In [45]:
# Demonstrations
esde = ESDE()
algebraic_result = await esde.solve("Prove that the sum of the first n odd numbers is n squared")
geometric_result = await esde.solve("Find the area of a triangle with vertices (0,0,0), (1,0,0), (0,1,0)")
print("Algebraic Result:", algebraic_result)
print("Geometric Result:", geometric_result)

ConnectionError: Error 111 connecting to localhost:6379. Connection refused.